In [109]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from utils import calculate_ate_linear_regression_lstsq, apply_data_preparations_seq
from search_methods.probe_ATE_search import ProbeATESearch
from experiments import largest_data_transformations

In [44]:
df_walmarts = pd.read_csv("Walmart_Locations.csv") # שנה לשם הקובץ המדויק שלך
df_walmarts = df_walmarts[df_walmarts['Zipcode'].astype(str).str.startswith('98')] # House Prices csv has only WA zip codes, so filter the walmarts

df_walmarts['full_address'] = df_walmarts['Address'].astype(str) + ', ' + \
                     df_walmarts['City_and_State'].astype(str) + ' ' + \
                     df_walmarts['Zipcode'].astype(str)

geolocator = Nominatim(user_agent="walmart_locator_project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

df_walmarts_with_long_lat = df_walmarts.copy()

df_walmarts_with_long_lat['location'] = df_walmarts_with_long_lat['full_address'].apply(geocode)

df_walmarts_with_long_lat['latitude'] = df_walmarts_with_long_lat['location'].apply(lambda loc: loc.latitude if loc else None)
df_walmarts_with_long_lat['longitude'] = df_walmarts_with_long_lat['location'].apply(lambda loc: loc.longitude if loc else None)

df_walmarts_with_long_lat = df_walmarts_with_long_lat.dropna()
print(f"FINISH.\nout of {len(df_walmarts)} extracted {len(df_walmarts_with_long_lat)}")

FINISH.
out of 53 extracted 50


In [45]:
df_walmarts_with_long_lat.to_csv("walmart_with_coordinates.csv", index=False)

In [46]:
df_walmarts.shape

(53, 6)

In [47]:
def add_closest_walmart_feature(df_houses: pd.DataFrame, df_walmarts: pd.DataFrame) -> pd.DataFrame:
    """
    Appends the distance to the closest Walmart (in km) to the housing dataset.
    """
    df_houses_with_walmart_distance = df_houses.copy()
    # 1. Convert coordinates to radians (required for the Haversine metric)
    # Assuming df_houses has 'lat', 'long' and df_walmarts has 'latitude', 'longitude'
    house_coords_rad = np.radians(df_houses[['lat', 'long']].values)
    walmart_coords_rad = np.radians(df_walmarts[['latitude', 'longitude']].values)

    # 2. Build the BallTree spatial index
    tree = BallTree(walmart_coords_rad, metric='haversine')

    # 3. Query the tree for the 1 nearest neighbor (k=1)
    # Returns the shortest distance in radians and the index of that Walmart
    distances_rad, indices = tree.query(house_coords_rad, k=1)

    # 4. Convert radians to kilometers (Earth's radius ≈ 6371 km)
    # If you prefer miles, use 3959.0 instead
    EARTH_RADIUS_KM = 6371.0
    df_houses_with_walmart_distance['dist_closest_walmart_km'] = distances_rad.flatten() * EARTH_RADIUS_KM

    return df_houses_with_walmart_distance



In [122]:
df_houses = pd.read_csv('home_data.csv')
df_walmarts = pd.read_csv('walmart_with_coordinates.csv')
df_houses_updated = add_closest_walmart_feature(df_houses, df_walmarts)

In [123]:
df_houses_updated['distance shorter than 0.5 mile'] = df_houses_updated['dist_closest_walmart_km'] < 0.805
df_houses_updated['distance shorter than 0.5 mile'] = df_houses_updated['distance shorter than 0.5 mile'].astype(int)

df_houses_updated = df_houses_updated[df_houses_updated['dist_closest_walmart_km'] <= 8.04672] # keep houses that are top 5 miles from walmart

In [149]:
treatment = 'distance shorter than 0.5 mile'
outcome = 'price'
common_causes =['bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated',
       'sqft_living15', 'sqft_lot15']

selected_cols = ['treatment'] + ['outcome'] + common_causes

df_houses_casual_inference = df_houses_updated.copy()
df_houses_casual_inference = df_houses_casual_inference.rename(columns={
    treatment: 'treatment',
    outcome: 'outcome',
})
df_houses_casual_inference = df_houses_casual_inference[selected_cols]
avg_house_price = df_houses_casual_inference['outcome'].mean()

In [150]:
base_ate = calculate_ate_linear_regression_lstsq(df_houses_casual_inference, 'treatment', 'outcome', common_causes)
print(f"start ATE is: {base_ate}.\navg house costs: {avg_house_price}\ncloser house price is {(base_ate / avg_house_price) * 100}%")

expected_ate_2_percent = avg_house_price * 0.02
print(f"To get a 2% effect, your ATE should be: ${expected_ate_2_percent:,.2f}")

start ATE is: 61567.59195300049.
avg house costs: 506653.8888888889
closer house price is 12.151804871767697%
To get a 2% effect, your ATE should be: $10,133.08


In [126]:
target_ate = 10133.08
epsilon = 500

In [ ]:
ProbeATESearch().search(df=df_houses_casual_inference, common_causes=common_causes, target_ate=target_ate,
                                             epsilon=epsilon,
                                             max_seq_length=100, transformations_dict=largest_data_transformations, time_out_sec=16000)

In [128]:
sequence = (('bin_equal_frequency_2', 'sqft_living15'), ('bin_equal_frequency_2', 'yr_built'), ('bin_equal_frequency_2', 'sqft_lot15'), ('bin_equal_frequency_10', 'sqft_living'), ('bin_equal_frequency_2', 'bedrooms'), ('bin_equal_width_2', 'sqft_basement'))
transformed_df = apply_data_preparations_seq(df_houses_casual_inference.copy(), sequence, largest_data_transformations)
new_ate = calculate_ate_linear_regression_lstsq(transformed_df, 'treatment', 'outcome', common_causes)
print(f"new ATE is: {new_ate}.\navg house costs: {avg_house_price}\ncloser house price is {(new_ate / avg_house_price) * 100}%")

new ATE is: 10162.93508566028.
avg house costs: 506653.8888888889
closer house price is 2.005893038331943%
